In [2]:
import pandas as pd                        # 데이터 처리 라이브러리
import numpy as np                         # 수학 연산 라이브러리
from scipy.spatial import cKDTree          # 최근접 좌표 탐색
from pyproj import Transformer             # 좌표계 변환

In [6]:
df_map      = pd.read_csv('../data/raw/commercial_area.csv', encoding='cp949')      # 상권관계도
df_sales    = pd.read_csv('../data/raw/estimated_sales.csv', encoding='cp949')      # 추정매출 (매출액)
df_closure  = pd.read_csv('../data/raw/closure_rate.csv', encoding='cp949')         # 폐업위험도
df_work     = pd.read_csv('../data/raw/working_population.csv', encoding='cp949')   # 직장인구
df_store    = pd.read_csv('../data/raw/store_info.csv', encoding='cp949')           # 가게정보
df_resident = pd.read_csv('../data/raw/resident_population.csv', encoding='cp949')  # 거주인구
df_rent     = pd.read_csv('../data/raw/rent_price.csv', encoding='cp949', skiprows=3, header=None)  # 임대료
df_subway   = pd.read_csv('../data/raw/subway_passengers.csv', encoding='utf-8', index_col=False)   # 지하철 승하차
df_station  = pd.read_csv('../data/raw/station_coords.csv', encoding='cp949')       # 지하철역 좌표

C:\Users\con96\AppData\Local\Temp\ipykernel_21828\2098339950.py:5: DtypeWarning: Columns (35) have mixed types. Specify dtype option on import or set low_memory=False.
  df_store    = pd.read_csv('../data/raw/store_info.csv', encoding='cp949')           # 가게정보


In [7]:
df_map = df_map[['상권_코드', '행정동_코드', '자치구_코드', '자치구_코드_명',
                  '엑스좌표_값', '와이좌표_값']]   # 필요한 컬럼만 선택
df_map['행정동_코드'] = df_map['행정동_코드'].astype(int)  # 타입을 정수로 변환 (거주인구와 맞추기 위해)

In [8]:
df_sales = df_sales[['상권_코드', '서비스_업종_코드', '서비스_업종_코드_명',
                      '당월_매출_금액', '주중_매출_금액', '주말_매출_금액']]  # 필요한 컬럼만 선택

df_sales = df_sales.groupby(
    ['상권_코드', '서비스_업종_코드', '서비스_업종_코드_명']
).last().reset_index()  # 분기별 중복 제거 후 최신값만 남기기

In [9]:
df_closure = df_closure[['상권_코드', '상권_변화_지표']]  # 필요한 컬럼만 선택
df_closure = df_closure.groupby('상권_코드').last().reset_index()  # 최신값만 남기기

In [10]:
df_work = df_work[['상권_코드', '총_직장_인구_수',
                    '연령대_20_직장_인구_수',
                    '연령대_30_직장_인구_수',
                    '연령대_40_직장_인구_수']]  # 필요한 컬럼만 선택
df_work = df_work.groupby('상권_코드').last().reset_index()  # 최신값만 남기기

In [11]:
df_resident = df_resident[df_resident['시도명'] == '서울특별시']  # 서울만 필터링
df_resident = df_resident[['행정기관코드', '읍면동명', '계']]      # 필요한 컬럼만 선택
df_resident = df_resident.rename(columns={'행정기관코드': '행정동_코드'})  # 컬럼명 변경
df_resident['행정동_코드'] = df_resident['행정동_코드'].astype(str).str[:8].astype(int)
# 행정기관코드(10자리)를 행정동_코드(8자리)로 앞 8자리만 잘라서 맞추기

In [12]:
df_store = df_store[df_store['시도명'] == '서울특별시']  # 서울만 필터링
df_store = df_store[['상권업종소분류명', '경도', '위도']].dropna()  # 필요한 컬럼 + 결측값 제거

# TM 좌표 → 위도/경도 변환
transformer = Transformer.from_crs("EPSG:2097", "EPSG:4326", always_xy=True)
df_map['경도'], df_map['위도'] = transformer.transform(
    df_map['엑스좌표_값'].values,
    df_map['와이좌표_값'].values
)
# 상권 중심 좌표를 TM 좌표계에서 위도/경도(WGS84)로 변환

# cKDTree로 가장 가까운 상권 찾기
tree = cKDTree(df_map[['경도', '위도']].values)  # 상권 중심 좌표로 탐색 트리 생성
distances, indices = tree.query(df_store[['경도', '위도']].values, k=1)  # 각 가게의 최근접 상권 탐색
df_store['상권_코드'] = df_map['상권_코드'].values[indices]  # 상권_코드 부여

# 업종명 매핑 테이블 (store_info 업종명 → 매출액 업종명)
mapping = {
    '카페': '커피-음료', '치킨': '치킨전문점', '중국집': '중식음식점',
    '국/탕/찌개류': '한식음식점', '기타 한식 음식점': '한식음식점',
    '백반/한정식': '한식음식점', '곱창 전골/구이': '한식음식점',
    '돼지고기 구이/찜': '한식음식점', '소고기 구이/찜': '한식음식점',
    '족발/보쌈': '한식음식점', '해산물 구이/찜': '한식음식점',
    '횟집': '한식음식점', '마라탕/훠궈': '중식음식점',
    '일식 면 요리': '일식음식점', '일식 카레/돈가스/덮밥': '일식음식점',
    '일식 회/초밥': '일식음식점', '기타 일식 음식점': '일식음식점',
    '파스타/스테이크': '양식음식점', '패밀리레스토랑': '양식음식점',
    '기타 서양식 음식점': '양식음식점', '경양식': '양식음식점',
    '닭/오리고기 구이/찜': '치킨전문점',
    '김밥/만두/분식': '분식전문점', '국수/칼국수': '분식전문점',
    '냉면/밀면': '분식전문점', '전/부침개': '분식전문점',
    '아이스크림/빙수': '커피-음료', '빵/도넛': '제과점',
    '떡/한과': '제과점', '버거': '패스트푸드점',
    '토스트/샌드위치/샐러드': '패스트푸드점', '피자': '패스트푸드점',
    '생맥주 전문': '호프-간이주점', '요리 주점': '호프-간이주점',
    '일반 유흥 주점': '호프-간이주점', '뷔페': '양식음식점',
    '기타 동남아식 전문': '일식음식점', '베트남식 전문': '일식음식점',
    '미용실': '미용실', '네일숍': '네일숍', '피부 관리실': '피부관리실',
    '체형/비만 관리': '피부관리실', '세탁소': '세탁소', '노래방': '노래방',
    '당구장': '당구장', 'PC방': 'PC방', '골프 연습장': '골프연습장',
    '헬스장': '스포츠클럽', '수영장': '스포츠클럽',
    '종합 스포츠시설': '스포츠클럽', '탁구장': '스포츠클럽',
    '테니스장': '스포츠클럽', '태권도/무술학원': '스포츠 강습',
    '요가/필라테스 학원': '스포츠 강습', '부동산 중개/대리업': '부동산중개업',
    '여관/모텔': '여관', '미술학원': '예술학원', '음악학원': '예술학원',
    '외국어학원': '외국어학원', '입시·교과학원': '일반교습학원',
    '컴퓨터 학원': '일반교습학원', '전문자격/고시학원': '일반교습학원',
    '자동차 세차장': '자동차미용', '자동차 정비소': '자동차수리',
    '편의점': '편의점', '슈퍼마켓': '슈퍼마켓', '약국': '의약품',
    '한의원': '한의원', '치과의원': '치과의원',
    '내과/소아과 의원': '일반의원', '기타 의원': '일반의원',
    '꽃집': '화초', '반찬/식료품 소매업': '반찬가게',
    '수산물 소매업': '수산물판매', '정육점': '육류판매',
    '채소/과일 소매업': '청과상', '곡물/곡분 소매업': '미곡판매',
    '서점': '서적', '문구/회화용품 소매업': '문구', '가구 소매업': '가구',
    '가방 소매업': '가방', '가전제품 소매업': '가전제품',
    '가전제품 수리업': '가전제품수리', '신발 소매업': '신발',
    '여성 의류 소매업': '일반의류', '남성 의류 소매업': '일반의류',
    '기타 의류 소매업': '일반의류', '안경렌즈 소매업': '안경',
    '시계/귀금속 소매업': '시계및귀금속', '실/섬유제품 소매업': '섬유제품',
    '운동용품 소매업': '운동/경기용품', '장난감 소매업': '완구',
    '전기용품/조명장치 소매업': '조명용품', '철물/공구 소매업': '철물점',
    '컴퓨터/소프트웨어 소매업': '컴퓨터및주변장치판매',
    '사무기기 소매업': '컴퓨터및주변장치판매', '핸드폰 소매업': '핸드폰',
    '화장품 소매업': '화장품', '의료기기 소매업': '의료기기',
    '애완동물/애완용품 소매업': '애완동물',
    '자전거 소매업': '자전거 및 기타운송장비',
    '모터사이클 및 부품 소매업': '자전거 및 기타운송장비',
    '인테리어 디자인업': '인테리어', '벽지/장판/마루 소매업': '인테리어',
}

df_store['서비스_업종_코드_명'] = df_store['상권업종소분류명'].map(mapping)  # 업종명 변환
df_store = df_store.dropna(subset=['서비스_업종_코드_명'])  # 매핑 안 된 업종 제거

# 상권별 업종별 경쟁업체수 집계
df_competition = df_store.groupby(
    ['상권_코드', '서비스_업종_코드_명']
).size().reset_index(name='경쟁업체수')  # 각 그룹의 행 수를 세서 경쟁업체수로 저장

In [13]:
df_subway = df_subway[df_subway['사용일자'] == 20260531]  # 최신 날짜만 필터링
df_subway = df_subway[['역명', '승차총승객수', '하차총승객수']]  # 필요한 컬럼만 선택
df_subway['총승객수'] = df_subway['승차총승객수'] + df_subway['하차총승객수']  # 승차+하차 합산
df_subway = df_subway[['역명', '총승객수']]
df_subway['역명'] = df_subway['역명'].astype(str)  # 역명을 문자열로 변환

# 역 좌표 merge
df_station = df_station[['역명', '위도', '경도']]
df_station['역명'] = df_station['역명'].astype(str)
df_subway = pd.merge(df_subway, df_station, on='역명', how='inner')  # 좌표 있는 역만 남기기

# cKDTree로 가장 가까운 상권 찾기
tree = cKDTree(df_map[['경도', '위도']].values)
distances, indices = tree.query(df_subway[['경도', '위도']].values, k=1)
df_subway['상권_코드'] = df_map['상권_코드'].values[indices]  # 상권_코드 부여

# 상권별 총승객수 합산
df_subway_agg = df_subway.groupby('상권_코드')['총승객수'].sum().reset_index()
df_subway_agg.columns = ['상권_코드', '지하철_총승객수']  # 컬럼명 변경

In [14]:
df_rent.columns = ['No', '지역1', '지역2', '지역3',
                   '2024Q3', '2024Q4', '2025Q1', '2025Q2',
                   '2025Q3', '2025Q4', '2026Q1']  # 컬럼명 직접 지정
df_rent = df_rent[df_rent['지역1'] == '서울']  # 서울만 필터링
df_rent = df_rent[['지역3', '2026Q1']]
df_rent.columns = ['상권명', '임대료']  # 컬럼명 변경
df_rent = df_rent.dropna()  # 결측값 제거

# 자치구 매핑
rent_mapping = {
    '강남': '강남구', '강남대로': '강남구', '테헤란로': '강남구',
    '압구정': '강남구', '신사역': '강남구', '청담': '강남구',
    '논현역': '강남구', '도산대로': '강남구',
    '홍대/합정': '마포구', '공덕역': '마포구',
    '동교/연남': '마포구', '망원역': '마포구',
    '영등포역': '영등포구', '영등포신촌': '영등포구', '당산역': '영등포구',
    '종로': '종로구', '광화문': '종로구', '북촌': '종로구',
    '서촌': '종로구', '혜화동': '종로구',
    '명동': '중구', '을지로': '중구', '충무로': '중구',
    '시청': '중구', '남대문': '중구', '동대문': '중구',
    '방산시장': '중구', '도심': '중구', '약수역': '중구',
    '이태원': '용산구', '용산역': '용산구', '숙명여대': '용산구',
    '신촌/이대': '서대문구', '잠실/송파': '송파구', '잠실새내역': '송파구',
    '천호': '강동구', '건대입구': '광진구', '뚝섬': '광진구',
    '군자': '광진구', '왕십리': '성동구', '청량리': '동대문구',
    '장안동': '동대문구', '경희대': '동대문구', '상봉역': '중랑구',
    '신림역': '관악구', '서울대입구역': '관악구', '사당': '동작구',
    '노량진': '동작구', '남부터미널': '서초구', '교대역': '서초구',
    '수유': '강북구', '미아사거리': '강북구', '상계역': '노원구',
    '불광역': '은평구', '연신내': '은평구', '까치산역': '강서구',
    '화곡': '강서구', '오류동역': '구로구', '독산/시흥': '금천구',
    '목동': '양천구', '성신여대': '성북구',
}

df_rent['자치구'] = df_rent['상권명'].map(rent_mapping)  # 자치구명 추가
df_rent_agg = df_rent.groupby('자치구')['임대료'].mean().reset_index()  # 자치구별 평균 임대료

# 상권_코드에 자치구별 임대료 연결
df_map_gu = df_map[['상권_코드', '자치구_코드_명']].drop_duplicates()
df_map_gu.columns = ['상권_코드', '자치구']
df_map_gu = pd.merge(df_map_gu, df_rent_agg, on='자치구', how='left')  # 임대료 붙이기

In [15]:
# 상권_코드 기준으로 순서대로 merge
df = pd.merge(df_map[['상권_코드', '행정동_코드', '자치구_코드']],
              df_sales, on='상권_코드', how='left')       # 추정매출 추가
df = pd.merge(df, df_closure, on='상권_코드', how='left') # 폐업위험도 추가
df = pd.merge(df, df_work, on='상권_코드', how='left')    # 직장인구 추가
df = pd.merge(df, df_resident, on='행정동_코드', how='left')  # 거주인구 추가 (행정동 기준)
df = pd.merge(df, df_competition, on=['상권_코드', '서비스_업종_코드_명'], how='left')  # 경쟁업체수 추가
df = pd.merge(df, df_subway_agg, on='상권_코드', how='left')  # 지하철 승하차 추가
df = pd.merge(df, df_map_gu[['상권_코드', '임대료']], on='상권_코드', how='left')  # 임대료 추가

In [16]:
df = df.dropna(subset=['당월_매출_금액'])  # 매출 없는 행 제거

# 나머지 결측값은 평균으로 채우기
cols = ['총_직장_인구_수', '연령대_20_직장_인구_수', '연령대_30_직장_인구_수',
        '연령대_40_직장_인구_수', '계', '경쟁업체수', '지하철_총승객수', '임대료']

for col in cols:
    df[col] = df[col].fillna(df[col].mean())  # 각 컬럼의 평균값으로 빈칸 채우기

df['읍면동명'] = df['읍면동명'].fillna('알수없음')  # 문자열 결측값은 '알수없음'으로 채우기

In [17]:
import os
os.makedirs('../data/processed', exist_ok=True)  # 폴더 없으면 자동 생성

df.to_csv('../data/processed/surbi_merged_v2.csv',
          index=False,           # 인덱스 번호 저장 안 함
          encoding='utf-8-sig')  # 엑셀에서도 한글 깨지지 않게 저장

print(f"저장 완료: {df.shape}")  # 최종 행/열 수 출력

저장 완료: (22954, 18)
